###                                               Module 8- Working with Databricks Filesystem – DBFS

Unity Catalog volume paths are supported even when DBFS root is disabled. in free edition DBFS is not supported.

#### 8.2-DBFS commands – mkdirs , cp , mv , head, put, rm , 

#### 1.	mkdirs(path) — Create directories
* Creates a new folder (and parent folders if needed).




In [0]:
%sql
show volumes

database,volume_name
default,first
default,my_files


In [0]:
%sql
CREATE VOLUME workspace.default.new_volume_test;

new_volume created in unity catalog.

In [0]:
dbutils.fs.mkdirs("/Volumes/workspace/default/new_volume/new_folder")


True

In [0]:
display(dbutils.fs.ls("/Volumes/workspace/default/new_volume"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/new_volume/new_folder/,new_folder/,0,1790428902004


#### 2. put(path, contents, overwrite) — Write a small file
* Writes text content to a file in DBFS.



In [0]:
dbutils.fs.put("/Volumes/workspace/default/new_volume/hello.txt", " Hello world from DBFS", overwrite=True)

Wrote 22 bytes.


True

In [0]:
dbutils.fs.put("/Volumes/workspace/default/new_volume/new_folder/hello.txt", "Hello from DBFS!", overwrite=True)

Wrote 16 bytes.


True

#### 3. head(path) — View file content
* Shows the first few lines (default 1 KB) of a file.



In [0]:
dbutils.fs.head("/Volumes/workspace/default/new_volume/hello.txt")

' Hello world from DBFS'

#### 4. cp(source, destination, recurse=False) — Copy files or folders
* Copies file or directory.


In [0]:
# Copy a single file
dbutils.fs.cp("/Volumes/workspace/default/new_volume/hello.txt", "/Volumes/workspace/default/new_volume/hello_copy.txt")


True

In [0]:
# Copy a folder and its contents (recursive)



In [0]:
dbutils.fs.cp(
    "/Volumes/workspace/default/new_volume/new_folder",
    "/Volumes/workspace/default/new_volume/myfolder_backup",
    recurse=True
)

True

#### 5. mv(source, destination, recurse=False) — Move files or folders
* Moves/renames a file or folder



In [0]:
dbutils.fs.put("/Volumes/workspace/default/new_volume/new_folder/hello.txt", " Hello world from DBFS", overwrite=True)

Wrote 22 bytes.


True

In [0]:
# Move (rename) file
dbutils.fs.mv("/Volumes/workspace/default/new_volume/new_folder/hello.txt", "/Volumes/workspace/default/new_volume/new_folder/renamed_hello.txt")

True

In [0]:
# Move folders recursively



In [0]:
dbutils.fs.mkdirs("/Volumes/workspace/default/new_volume/new_folder/myfolder")

True

In [0]:
dbutils.fs.mkdirs("/Volumes/workspace/default/new_volume/new_folder/myfolder_archive")

True

In [0]:
dbutils.fs.mv("/Volumes/workspace/default/new_volume/new_folder/myfolder", "/Volumes/workspace/default/new_volume/new_folder/myfolder_archive", recurse=True)

True

#### 6. rm(path, recurse=False) — Delete a file
* Removes a file (or folder with recurse=True).



In [0]:

# Remove a single file
dbutils.fs.rm("/Volumes/workspace/default/new_volume/new_folder/hello_copy.txt")

False

In [0]:
# Remove a folder and all its contents
dbutils.fs.rm("/Volumes/workspace/default/new_volume/new_folder/myfolder", recurse=True)

False

#### 7. rmdir(path) — Remove an empty directory
* Deletes a folder only if it is empty.



In [0]:
dbutils.fs.rmdir("/Volumes/workspace/default/new_volume/new_folder/myfolder_archive", recurse=True)

# Will fail if the folder has files inside. Use rm(..., recurse=True) instead to force delete.

#### 8.3-How to handle multiple files in DBFS
* Let’s say you have multiple CSV files in a folder /FileStore/data/ in DBFS and you want to:
* 1.	List them
* 2.	Read all into one DataFrame
* 3.	Process or display them



In [0]:
* Step-by-step Simple Example
*🔹 1. Upload multiple files to DBFS folder

* 2.Read all CSV files into a single DataFrame
* If all files have the same schema, use a wildcard *.csv:


In [0]:
## display(dbutils.fs.ls("/FileStore/data/"))
## Free edition of DBX no DBFS available. Unity Catalog volume paths are supported even when DBFS root is disabled.
print("OK")

OK


In [0]:
%sql
show catalogs;

catalog
samples
sql_catalog
system
workspace


In [0]:
%sql
SHOW SCHEMAS IN workspace;

databaseName
default
information_schema
sql_schema


In [0]:
%sql
SHOW VOLUMES IN workspace.default;

database,volume_name
default,first
default,my_files
default,new_volume
default,new_volume_test


In [0]:
%sql
CREATE VOLUME workspace.default.db_volume;

In [0]:
display(dbutils.fs.ls("/Volumes/workspace/default/db_volume/"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/db_volume/Employees.csv,Employees.csv,57511,1793293826000
dbfs:/Volumes/workspace/default/db_volume/command_directory/,command_directory/,0,1793293908503


In [0]:
df = spark.read.option("header", "true").csv("/Volumes/workspace/default/db_volume/command_directory/*.csv")
display(df)
# This merges all CSV files into one DataFrame.


CustomerKey,Prefix,FirstName,LastName,BirthDate,MaritalStatus,Gender,EmailAddress,AnnualIncome,TotalChildren,EducationLevel,Occupation,HomeOwner
11000,MR.,JON,YANG,4/8/1966,M,M,jon24@adventure-works.com,"$90,000",2,Bachelors,Professional,Y
11001,MR.,EUGENE,HUANG,5/14/1965,S,M,eugene10@adventure-works.com,"$60,000",3,Bachelors,Professional,N
11002,MR.,RUBEN,TORRES,8/12/1965,M,M,ruben35@adventure-works.com,"$60,000",3,Bachelors,Professional,Y
11003,MS.,CHRISTY,ZHU,2/15/1968,S,F,christy12@adventure-works.com,"$70,000",0,Bachelors,Professional,N
11004,MRS.,ELIZABETH,JOHNSON,8/8/1968,S,F,elizabeth5@adventure-works.com,"$80,000",5,Bachelors,Professional,Y
11005,MR.,JULIO,RUIZ,8/5/1965,S,M,julio1@adventure-works.com,"$70,000",0,Bachelors,Professional,Y
11007,MR.,MARCO,MEHTA,5/9/1964,M,M,marco14@adventure-works.com,"$60,000",3,Bachelors,Professional,Y
11008,MRS.,ROBIN,VERHOFF,7/7/1964,S,F,rob4@adventure-works.com,"$60,000",4,Bachelors,Professional,Y
11009,MR.,SHANNON,CARLSON,4/1/1964,S,M,shannon38@adventure-works.com,"$70,000",0,Bachelors,Professional,N
11010,MS.,JACQUELYN,SUAREZ,2/6/1964,S,F,jacquelyn20@adventure-works.com,"$70,000",0,Bachelors,Professional,N


In [0]:
# 3. List all files in a DBFS folder
files = dbutils.fs.ls("/Volumes/workspace/default/db_volume/command_directory/")
# print(files)
for file in files:
    print(file.name)

Customers.csv


#### 8.4-Loading data in Databricks
* 1. Upload and Load File using DBFS (UI Method)
* 🔹 Step 1: Upload via UI
*	In Databricks, go to Data > DBFS > Upload
*	Upload your file (CSV, Excel, JSON, etc.)
*	It will be stored under: /FileStore/tables/filename.csv



/* *🔹 Step 2: Read into DataFrame

df = spark.read.option("header", True).csv("/Volumes/workspace/default/new_volume/new_folder/filename.csv")
df.show()